# Digikala Dataset Exploration — Phase 1

This notebook introduces the structure, quality, and relationship of the two datasets. The production profiling logic in `src/digikala_llm/eda.py` remains the tested source of truth. Interactive analysis uses **bounded samples of at most 100,000 rows from each file**. One later join check scans the products file in chunks while retaining only rows relevant to the bounded comments sample. The first 100,000 rows are convenient for exploration, but they are not a random sample and may not be statistically representative.

## 1. Libraries and project paths
The project root is detected so the notebook works when opened from either the repository root or the `notebooks` directory.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from digikala_llm.eda import detect_format

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
COMMENTS_PATH = DATA_DIR / 'digikala-comments.csv'
PRODUCTS_PATH = DATA_DIR / 'digikala-products.csv'
REPORT_PATH = PROJECT_ROOT / 'reports' / 'eda' / 'dataset_profile.json'
SAMPLE_ROWS = 100_000
assert 0 < SAMPLE_ROWS <= 100_000
sns.set_theme(style='whitegrid')
PROJECT_ROOT, COMMENTS_PATH, PRODUCTS_PATH

## 2. Detect the encoding and delimiter
We reuse the tested `detect_format` function from the production module instead of duplicating its logic. It reads only a bounded sample from the beginning of each file.

In [ ]:
for path in (COMMENTS_PATH, PRODUCTS_PATH):
    if not path.is_file():
        raise FileNotFoundError(f'Dataset file not found: {path}')

detected_comments_encoding, comments_delimiter = detect_format(COMMENTS_PATH)
products_encoding, products_delimiter = detect_format(PRODUCTS_PATH)
comments_encoding = 'utf-8-sig'  # Both source files have a UTF-8 BOM.
if detected_comments_encoding != comments_encoding:
    raise ValueError(f'Unexpected comments encoding: {detected_comments_encoding!r}')
pd.DataFrame({
    'dataset': ['comments', 'products'],
    'encoding': [comments_encoding, products_encoding],
    'delimiter': [repr(comments_delimiter), repr(products_delimiter)],
})

## 3. Load bounded samples safely
The `nrows` argument provides an explicit limit, so neither complete dataset is loaded into memory.

In [ ]:
comments = pd.read_csv(COMMENTS_PATH, encoding='utf-8-sig', sep=comments_delimiter, nrows=SAMPLE_ROWS, low_memory=False)
products = pd.read_csv(PRODUCTS_PATH, encoding=products_encoding, sep=products_delimiter, nrows=SAMPLE_ROWS, low_memory=False)
assert len(comments) <= SAMPLE_ROWS and len(products) <= SAMPLE_ROWS
EXPECTED_COMMENT_COLUMNS = [
    'id', 'title', 'body', 'created_at', 'rate', 'recommendation_status',
    'is_buyer', 'product_id', 'advantages', 'disadvantages', 'likes',
    'dislikes', 'seller_title', 'seller_code', 'true_to_size_rate',
]
if 'id' not in comments.columns:
    raise ValueError("The comments sample is missing the expected 'id' column. Check CSV decoding.")
bom_columns = [column for column in comments.columns if '\ufeff' in str(column) or 'ï»' in str(column)]
if bom_columns:
    raise ValueError(f'BOM bytes remain in comment column names: {bom_columns}')
if comments.columns.tolist() != EXPECTED_COMMENT_COLUMNS:
    raise ValueError(f'Unexpected comment columns: {comments.columns.tolist()}')
text_preview = ' '.join(comments.select_dtypes(include='object').head(1_000).fillna('').astype(str).to_numpy().ravel())
mojibake_markers = ('ï»', 'Ã', 'Â', 'Ø', 'Ù', 'ظ¾', 'ط§', 'غŒ')
found_markers = [marker for marker in mojibake_markers if marker in text_preview]
if found_markers:
    raise ValueError(f'Possible mojibake found in the comments sample: {found_markers}')
print(f'comments sample: {comments.shape}')
print(f'products sample: {products.shape}')

## 4. Shapes, column names, data types, and sample rows
These checks provide a quick overview of each sample. Remember that pandas infers data types from the sample, so types may differ elsewhere in the complete dataset.

In [ ]:
display(pd.DataFrame({'dataset': ['comments', 'products'], 'rows': [len(comments), len(products)], 'columns': [comments.shape[1], products.shape[1]]}))
print('Comments columns:', comments.columns.tolist())
print('Products columns:', products.columns.tolist())
display(comments.dtypes.rename('dtype').to_frame())
display(products.dtypes.rename('dtype').to_frame())
display(comments.head(3))
display(products.head(3))

## 5. Missing values
For every column, we calculate both the number and percentage of missing values in the sample.

In [ ]:
def missing_summary(frame):
    counts = frame.isna().sum()
    return pd.DataFrame({'missing_count': counts, 'missing_percent': (counts / len(frame) * 100).round(2)}).sort_values('missing_count', ascending=False)

display(missing_summary(comments))
display(missing_summary(products))

## 6. Duplicate IDs in each sample
Comments and products are evaluated separately. `exact_full_row_duplicate_count` counts repeated complete rows beyond their first occurrence. `duplicate_id_excess_row_count` counts repeated ID rows beyond the first row for each ID. `duplicated_unique_id_count` counts affected non-null IDs once. These results describe only the first 100,000 rows. A product ID is a product grouping key, not a unique row key: the raw products file appears to mix product facts with seller offers.

In [ ]:
def duplicate_summary(frame):
    duplicate_id_mask = frame['id'].notna() & frame['id'].duplicated(keep=False)
    return {
        'missing_id_count': int(frame['id'].isna().sum()),
        'exact_full_row_duplicate_count': int(frame.duplicated().sum()),
        'duplicate_id_excess_row_count': int(frame.loc[frame['id'].notna(), 'id'].duplicated().sum()),
        'duplicated_unique_id_count': int(frame.loc[duplicate_id_mask, 'id'].nunique()),
    }

comments_id_checks = duplicate_summary(comments)
products_id_checks = duplicate_summary(products)
id_checks = pd.DataFrame.from_dict(
    {'comments.id': comments_id_checks, 'products.id': products_id_checks}, orient='index'
)
display(id_checks)

duplicate_product_mask = products['id'].notna() & products['id'].duplicated(keep=False)
duplicate_product_columns = ['id', 'title_fa', 'Seller', 'Price', 'Rate', 'Rate_cnt']
duplicate_product_examples = (
    products.loc[duplicate_product_mask, duplicate_product_columns]
    .sort_values('id')
    .head(30)
)
display(duplicate_product_examples)

duplicate_product_rows = products.loc[duplicate_product_mask]
exact_duplicate_offer_count = int(duplicate_product_rows.duplicated().sum())
multiple_seller_ids = int(
    duplicate_product_rows.groupby('id')['Seller'].nunique(dropna=False).gt(1).sum()
)
different_price_ids = int(
    duplicate_product_rows.groupby('id')['Price'].nunique(dropna=False).gt(1).sum()
)
core_attribute_columns = [
    'title_fa', 'Category1', 'Category2', 'Brand', 'Rate', 'Rate_cnt', 'sub_category'
]
conflicting_attribute_ids = int(
    duplicate_product_rows.groupby('id')[core_attribute_columns]
    .nunique(dropna=False).gt(1).any(axis=1).sum()
)
duplicate_product_interpretation = pd.Series({
    'Exact duplicate offers beyond the first': exact_duplicate_offer_count,
    'Product IDs with multiple sellers': multiple_seller_ids,
    'Product IDs with different prices': different_price_ids,
    'Product IDs with conflicting core attributes': conflicting_attribute_ids,
}, name='count')
display(duplicate_product_interpretation.to_frame())

## 7. Rating and comment-attribute distributions
First, we convert ratings to numeric values safely. Invalid values become `NaN`, which allows us to identify and count them separately.

In [ ]:
comments_rate = pd.to_numeric(comments['rate'], errors='coerce')
products_rate = pd.to_numeric(products['Rate'], errors='coerce')
display(comments_rate.value_counts(dropna=False).sort_index().rename('comments.rate count').to_frame())
display(products_rate.value_counts(dropna=False).sort_index().rename('products.Rate count').to_frame())
display(comments['recommendation_status'].value_counts(dropna=False).rename('count').to_frame())
display(comments['is_buyer'].value_counts(dropna=False).rename('count').to_frame())
rate_recommendation_crosstab = pd.crosstab(
    comments_rate, comments['recommendation_status'], dropna=False, margins=True
)
display(rate_recommendation_crosstab)
display(rate_recommendation_crosstab.loc[[0]] if 0 in rate_recommendation_crosstab.index else pd.DataFrame())
products_rate_count = pd.to_numeric(products['Rate_cnt'], errors='coerce')
zero_rating_vs_count = pd.crosstab(
    products_rate.eq(0).map({True: 'Rate = 0', False: 'Rate != 0'}),
    products_rate_count.eq(0).map({True: 'Rate_cnt = 0', False: 'Rate_cnt != 0'}),
    margins=True,
)
display(zero_rating_vs_count)
display(products.loc[products_rate.eq(0), 'Rate_cnt'].value_counts(dropna=False).head(20).rename('count').to_frame())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sns.countplot(x=comments_rate, ax=axes[0, 0], color='#e53935')
axes[0, 0].set(title='Comment rating distribution', xlabel='comments.rate', ylabel='Count')
sns.histplot(products_rate.dropna(), bins=20, ax=axes[0, 1], color='#1976d2')
axes[0, 1].set(title='Product rating distribution', xlabel='products.Rate', ylabel='Count')
comments['recommendation_status'].value_counts().plot.bar(ax=axes[1, 0], color='#43a047')
axes[1, 0].set(title='Recommendation status', xlabel='recommendation_status', ylabel='Count')
comments['is_buyer'].value_counts().plot.bar(ax=axes[1, 1], color='#8e24aa')
axes[1, 1].set(title='Buyer status distribution', xlabel='is_buyer', ylabel='Count')
plt.tight_layout()

## 8. Product prices and invalid values
Missing, non-numeric, zero, and negative prices are reported separately. Descriptive statistics use numeric values only. The likely currency is **unknown and must not be assumed until the dataset documentation or source is verified**.

In [ ]:
price_raw = products['Price']
price = pd.to_numeric(price_raw, errors='coerce')
price_validation = pd.Series({
    'missing': int(price_raw.isna().sum()),
    'non_numeric': int((price_raw.notna() & price.isna()).sum()),
    'zero': int((price == 0).sum()),
    'negative': int((price < 0).sum()),
})
price_statistics = price.describe(percentiles=[0.5, 0.95, 0.99]).rename('Price')
display(price_statistics)
display(price_validation.rename('count').to_frame())
highest_price_rows = (
    products.assign(_numeric_price=price)
    .nlargest(10, '_numeric_price')
    [['id', 'title_fa', 'Seller', 'Price', 'Rate', 'Rate_cnt']]
)
display(highest_price_rows)
sns.histplot(price[price > 0], bins=40)
plt.title('Positive price distribution in the sample')
plt.xlabel('Price')
plt.ylabel('Count')

## 9. Most common product categories and brands
We display only the 15 most common values so the tables and charts remain readable.

In [ ]:
def find_column(frame, candidates):
    lookup = {str(column).lower(): column for column in frame.columns}
    return next((lookup[name.lower()] for name in candidates if name.lower() in lookup), None)

category_column = find_column(products, ['Category1', 'category', 'Category', 'category_title'])
brand_column = find_column(products, ['brand', 'Brand', 'brand_name'])
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for column, axis, title in [(category_column, axes[0], 'Top categories'), (brand_column, axes[1], 'Top brands')]:
    if column is None:
        axis.text(0.5, 0.5, 'No suitable column found', ha='center')
        axis.set_axis_off()
    else:
        counts = products[column].value_counts().head(15)
        display(counts.rename('count').to_frame())
        counts.sort_values().plot.barh(ax=axis)
        axis.set_title(title)
plt.tight_layout()

## 10. Illustrative sample join and bounded product lookup
The intended relationship is `products.id <- comments.product_id`. First, we show coverage between two independently truncated samples. This is only an illustrative sample join and **cannot estimate a true orphan rate**. Next, we keep the comments sample bounded at 100,000 rows and stream the products file in chunks. Only product rows whose IDs occur in the comments sample are retained.

In [ ]:
def normalized_ids(series):
    return series.astype('string').str.strip()

comment_product_ids = normalized_ids(comments['product_id'])
sample_product_ids = set(normalized_ids(products['id']).dropna())
illustrative_match_mask = comment_product_ids.isin(sample_product_ids)
illustrative_coverage = pd.Series({
    'comment rows with product_id': int(comment_product_ids.notna().sum()),
    'comment rows matching the independent product sample': int(illustrative_match_mask.sum()),
    'comment rows not covered by the independent product sample': int((comment_product_ids.notna() & ~illustrative_match_mask).sum()),
}, name='row_count')
display(illustrative_coverage.to_frame())

sampled_product_id_set = set(comment_product_ids.dropna())
matching_product_chunks = []
product_join_columns = ['id', 'title_fa', 'Seller', 'Price', 'Rate', 'Rate_cnt']
for product_chunk in pd.read_csv(
    PRODUCTS_PATH, encoding=products_encoding, sep=products_delimiter,
    usecols=product_join_columns, dtype=str, chunksize=50_000,
):
    product_chunk_ids = normalized_ids(product_chunk['id'])
    matching_rows = product_chunk.loc[product_chunk_ids.isin(sampled_product_id_set)].copy()
    if not matching_rows.empty:
        matching_rows['id'] = normalized_ids(matching_rows['id'])
        matching_product_chunks.append(matching_rows)

matching_products = (
    pd.concat(matching_product_chunks, ignore_index=True)
    if matching_product_chunks else pd.DataFrame(columns=product_join_columns)
)
matched_product_ids = set(matching_products['id'].dropna())
matched_comment_mask = comment_product_ids.isin(matched_product_ids)
product_seller_counts = matching_products.groupby('id')['Seller'].nunique(dropna=False)
bounded_join_results = pd.Series({
    'sampled comment rows matched to at least one product': int(matched_comment_mask.sum()),
    'sampled comment rows with no product match anywhere': int((comment_product_ids.notna() & ~matched_comment_mask).sum()),
    'sampled comment rows with missing product_id': int(comment_product_ids.isna().sum()),
    'unique sampled product IDs matched': len(matched_product_ids),
    'product IDs with multiple seller offers': int((product_seller_counts > 1).sum()),
}, name='count')
display(bounded_join_results.to_frame())
display(matching_products.head(10))

## 11. Interpretation notes and data-quality warnings
Use these rules in later cleaning and modeling:

- `comments.rate = 0` must not automatically be interpreted as negative sentiment because most zero-rate rows are recommended in this sample.
- `products.Rate = 0` together with `Rate_cnt = 0` should be treated as unrated or missing in later cleaning.
- The `Price` currency is currently unknown.
- Zero prices are invalid. High prices should be flagged for review, not automatically removed.
- `products.id` is not a unique row key; the raw file appears to combine product facts and seller offers.

The next cell summarizes warnings detected in the bounded samples. Use the complete report generated separately by the CLI before making final decisions.

In [ ]:
warnings = []
if id_checks['missing_id_count'].sum(): warnings.append('The samples contain missing IDs.')
if id_checks['duplicated_unique_id_count'].sum(): warnings.append('The samples contain duplicate IDs.')
if ((comments_rate < 0) | (comments_rate > 5)).any(): warnings.append('Some comments.rate values are outside the expected 0–5 range.')
if ((products_rate < 0) | (products_rate > 100)).any(): warnings.append('Some products.Rate values are outside the expected 0–100 range.')
if price_validation['missing']: warnings.append('The product sample contains missing prices.')
if price_validation['non_numeric']: warnings.append('The product sample contains non-numeric prices.')
if price_validation['zero']: warnings.append('The product sample contains zero prices.')
if price_validation['negative']: warnings.append('The product sample contains negative prices.')
if bounded_join_results['sampled comment rows with no product match anywhere']:
    warnings.append('Some sampled comments have no matching product anywhere in the products file.')
warnings or ['No clear data-quality warnings were found in the initial sample checks.']

## 12. Optional disabled section: load the complete JSON report
This section **does not run the complete profiler**. It only reads a previously generated report if one exists. To enable it intentionally, change `LOAD_FULL_REPORT` to `True`.

In [ ]:
LOAD_FULL_REPORT = False  # Intentionally disabled
full_report = None
if LOAD_FULL_REPORT:
    if not REPORT_PATH.is_file():
        raise FileNotFoundError(f'Previously generated report not found: {REPORT_PATH}')
    with REPORT_PATH.open(encoding='utf-8') as report_file:
        full_report = json.load(report_file)
    print('Existing report loaded; no new profiling was run.')
else:
    print('Complete report loading is disabled.')